In [1]:
PYTHON_BIN = "/content/venv/bin/python"

In [9]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv

# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip

# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,180 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages

In [10]:
# %% [markdown]
# # Week 3 shared Colab scaffold
#
# This file is the source of truth for the reusable Colab cells every week-3 lab
# uses. It is written in py-percent format: each `# %%` block is one standalone,
# pasteable Colab cell. Copy the cells you need into the day's notebook in the
# order the lab README gives.
#
# Why this scaffold exists: a Colab notebook runs cells one at a time, top to
# bottom, and a cell blocks until it returns. A live inference server does not
# return: it runs until you kill it. So you cannot "start the server" in one cell
# and "watch it" in the next the way you would in a terminal with two panes. The
# pattern here is launch-then-poll: one cell launches the server as a background
# subprocess and returns immediately, and a second cell polls the health endpoint
# until the server answers or a timeout fires. Every long-running piece (the
# server, the nvidia-smi sampler) runs in the background and is watched by a
# short cell that returns.
#
# Pin source: versions come from ../../../PINS.md (course root). The vLLM-on-T4 pin
# is verified on a real free-tier T4 before the cohort starts; the confirmed
# version and date land in PINS.md under "Verification status". Do not invent a
# vLLM version here; read the pin.
#
# Convert to a .ipynb when you want a notebook file (the .py stays the source of
# truth):
#   uvx jupytext --to ipynb colab_scaffold.py

# %%
# PINS block. These mirror ../../../PINS.md (course root, the single source of
# truth). If a pin changes, it changes in PINS.md first, then here. The vLLM pin
# is the load-bearing one: it must be the version confirmed on a real free-tier
# T4 during the pre-cohort verification pass. Read PINS.md before you run this.
#
# PINS (from ../../../PINS.md):
#   VLLM_PIN=0.6.*          # OpenAI server; runs the xformers backend on sm75
#   BITSANDBYTES_PIN=0.49.2 # int8/int4 load path (day 1 profiling); 0.44.* is
#                           # broken on Colab's cu128 torch, see PINS.md
#   AUTOAWQ_PIN=0.2.*       # AWQ weights load path (day 4)
#   TRANSFORMERS_PIN=4.46.* # streaming generation (day 2)
#   ACCELERATE_PIN=1.1.*    # device placement
#   HTTPX_PIN=0.27.*        # async A/B client (day 3)
#   OPENAI_PIN=1.54.*       # the client that proves the /v1 contract

# %%
# Cell: the pins and the installer function. Defines only, installs nothing.
# Paste this on every week-3 day. Then paste ONE of the two install cells below,
# whichever the day's README names. Day 1 profiles with transformers and must
# NOT install vLLM; days 2 to 5 serve, and must.
import subprocess, sys

# Pins mirrored from ../../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)


In [11]:
# %%
# INSTALL CELL B: the serving set. This is days 3 to 5 (day 2 is CELL A:
# direct transformers loads crash on vLLM's numpy - verified on T4 2026-08-07). About 30 minutes on a
# cold runtime, and it prints almost nothing for most of it, so start it and go
# and fill in your prediction card. vLLM brings its own torch; do NOT install a
# second one.
#
# transformers and accelerate are NOT optional here, even on days you never call
# them directly. vLLM 0.6.x installs its own torch (2.5.1), which downgrades
# Colab's torch and leaves Colab's preinstalled torchaudio compiled against the
# wrong ABI. Colab's preinstalled transformers imports torchaudio at module load,
# so vLLM then dies during startup with
#   OSError: _torchaudio.abi3.so: undefined symbol: aoti_torch_abi_version
# Pinning transformers to 4.46 removes that import path. Verified on a T4,
# 2026-07-27: without these two lines the server never comes up.
#
# autoawq is only needed on day 4; that README says so and adds it to this call.
# %%
# INSTALL CELL B: serving set for Days 3–5

PYTHON_BIN = "/content/venv/bin/python"

# Install the pinned serving packages inside the Python 3.10 virtual environment.
!{PYTHON_BIN} -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("serving pins installed")

serving pins installed


In [12]:

# %%
# Cell: launch the server as a background subprocess.
# vLLM's OpenAI-compatible server runs until killed, so it cannot live in a cell
# that must return. Popen launches it in the background and this cell returns at
# once. stdout and stderr are teed to /content/server.log so the health-poll cell
# and you can read what happened. Flags come from PINS.md canon: --dtype half is
# mandatory on the T4 (sm75 has no bf16 and no FlashAttention, so vLLM uses the
# xformers backend). Edit SERVER_ARGS for the day (day 3 is plain, day 4 adds
# --quantization awq and the tool-call flags).
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",                 # sm75: no bf16, no FlashAttention
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 4552, logging to /content/server.log


In [13]:
!nohup /content/venv/bin/python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --host 0.0.0.0 \
    --port 8000 \
    --dtype half \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.85 \
    > /content/vllm_server.log 2>&1 &

In [14]:
!tail -20 /content/vllm_server.log

INFO 09-01 13:13:56 worker.py:241] Memory profiling takes 1.47 seconds
INFO 09-01 13:13:56 worker.py:241] the current vLLM instance can use total_gpu_memory (14.56GiB) x gpu_memory_utilization (0.85) = 12.38GiB
INFO 09-01 13:13:56 worker.py:241] model weights take 2.89GiB; non_torch_memory takes 0.20GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 7.90GiB.
INFO 09-01 13:13:56 gpu_executor.py:76] # GPU blocks: 18480, # CPU blocks: 9362
INFO 09-01 13:13:56 gpu_executor.py:80] Maximum concurrency for 4096 tokens per request: 72.19x
INFO 09-01 13:14:02 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as need

In [15]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct","object":"model","created":1788268847,"owned_by":"vllm","root":"Qwen/Qwen2.5-1.5B-Instruct","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-98bdcfe7b2b54e39a52c0688e356573e","object":"model_permission","created":1788268847,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [16]:

# %%
# Cell: health poll.
# The launch cell returned immediately; the server is still loading weights in
# the background. This cell polls GET /v1/models until it answers 200 or the
# timeout fires. First launch on a fresh runtime downloads the model, so the
# first poll can take a while; that is what the 300s timeout is for. On timeout
# it prints the last 30 log lines so you can see why (usually still downloading,
# or an OOM, or a bad flag).
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 0s: http://localhost:8000/v1/models -> 200


In [17]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [23]:
import json

baseline = json.load(open("baselines.sample (1).json"))

print("baseline batch tokens/s:", baseline["batch"])

baseline batch tokens/s: {'1': 34.0, '4': 49.8, '8': 96.6}


In [24]:
from google.colab import files
uploaded = files.upload()   # pick baselines.json
import json
baseline = json.load(open("baselines.sample (1).json"))
print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.sample.json to baselines.sample (2).json
baseline batch tokens/s: {'1': 34.0, '4': 49.8, '8': 96.6}


In [25]:
# Async A/B client for Lab W3D3 (engine swap).
# Paste the whole file as one Colab cell (after the vLLM server is healthy), then
# call run_sweep(...) as the day-3 README shows. It fires N concurrent chat
# completions per level with httpx + asyncio, excludes a warm-up round, and
# reports aggregate tokens/s at each concurrency level.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same client works
# against any team's service. No secrets: the local vLLM server needs no key.

import asyncio
import time

import httpx

# A fixed prompt set so every run measures the same work. Varied lengths, no
# duplicates. Requests cycle through this list.
FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

# Output lengths per request, cycled in order. This list is IDENTICAL to Monday's
# QUEUE in the day-2 lab, and it has to stay that way: the A/B is only honest if
# both engines are asked for exactly the same work. 24 requests, 18 that want 32
# tokens and 6 that want 256, so a long request is always in flight alongside
# short ones.
#
# The mixed lengths are the entire point. Ask every request for the same number
# of tokens and there is no straggler, static batching pays no tax, and
# continuous batching has nothing to win back. You would measure a flat speedup
# across concurrency and conclude, wrongly, that continuous batching does not
# scale.
QUEUE = [32, 32, 32, 256] * 6

# Fallback when a caller does not pass a length.
MAX_TOKENS = 128
# Warm-up requests per level, dropped from the timing.
WARMUP = 4


async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    """Fire one chat completion, return the count of completion tokens."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    # completion_tokens is what the server generated; fall back to counting.
    # Accounting note vs Monday: static_queue counted REQUESTED tokens, which
    # equals generated there (greedy decode runs to the cap). vLLM can stop at
    # EOS short of the cap, so counting usage is the honest number for it -
    # any bias this introduces runs AGAINST vLLM, never for it.
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct


async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    """Run total_requests requests, at most `concurrency` in flight at once."""
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    # Each request carries its own output length from QUEUE, so the workload
    # matches Monday's static-batching baseline request for request.
    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                         QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    """Sweep the concurrency levels; return a list of per-level result dicts.

    A warm-up round runs first and is discarded so model-load and cache-warm
    cost stays out of the measured numbers.
    """
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        # warm-up: fire WARMUP requests, ignore timing
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results

prompts = FIXED_PROMPTS            # defined in ab_client.py
vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)
for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 37.7, 'wall_s': 36.835}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 160.0, 'wall_s': 8.68}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 221.0, 'wall_s': 6.284}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 37.7, 'wall_s': 36.835}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 160.0, 'wall_s': 8.68}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 221.0, 'wall_s': 6.284}


In [26]:
import json

def tokps_at(level_list, c):
    return next(x["tokens_per_s"] for x in level_list if x["concurrency"] == c)

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
# Monday's static batching at 1/4/8 is the baseline curve
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {c: round(vllm_by_c[c] / base_by_c[c], 2)
           for c in vllm_by_c if c in base_by_c}

report = {
    "baseline": base_by_c,               # from Monday's baselines.json
    "vllm": vllm_by_c,                    # measured today
    "speedup_by_concurrency": speedup,
    "predicted_speedup": None,           # <- put your prediction-card number here
}
with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 34.0,
    "4": 49.8,
    "8": 96.6
  },
  "vllm": {
    "1": 37.7,
    "4": 160.0,
    "8": 221.0
  },
  "speedup_by_concurrency": {
    "1": 1.11,
    "4": 3.21,
    "8": 2.29
  },
  "predicted_speedup": null
}


In [27]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]
print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 2.84x, vLLM scales 5.86x
continuous batching is worth 2.06x of scaling


In [28]:

# %%
# Cell: clean shutdown.
# Terminate the server process group and confirm port 8000 is free again. Run
# this between labs, or before relaunching with different flags. Killing only the
# parent pid can leave a child holding the port; killpg kills the whole group the
# launch cell created with start_new_session=True.
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    # give it a moment, then confirm the port is free
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")
shutdown_server()

sent SIGTERM to process group of pid 4552


In [29]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it. It reads
# ab_report.json and checks the schema, that vLLM's concurrency-8 throughput
# beats Monday's batch-8 baseline, and that the speedup fields were computed.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


baseline batch-8: 96.6, vllm concurrency-8: 221.0
speedup at 8: 2.29x
GREEN CHECK: PASS


EXTRA LAB

In [30]:
import asyncio, time, httpx

async def send_one(client, base_url, model, prompt, max_tokens=128):
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "temperature": 0.0}
    t0 = time.perf_counter()
    try:
        r = await client.post(f"{base_url}/chat/completions", json=payload)
        r.raise_for_status()
        return {"ok": True, "latency_s": time.perf_counter() - t0}
    except Exception as e:
        return {"ok": False, "latency_s": time.perf_counter() - t0, "error": str(e)}

def p95(latencies):
    s = sorted(latencies)
    idx = max(0, int(len(s) * 0.95) - 1)
    return s[idx]

async def naive_burst(base_url, model, prompt, n=50, max_tokens=128):
    async with httpx.AsyncClient(timeout=120.0) as client:
        results = await asyncio.gather(*[
            send_one(client, base_url, model, prompt, max_tokens) for _ in range(n)
        ])
    latencies = [r["latency_s"] for r in results if r["ok"]]
    return {
        "n_sent": n,
        "n_ok": len(latencies),
        "p95_s": round(p95(latencies), 3) if latencies else None,
        "mean_s": round(sum(latencies) / len(latencies), 3) if latencies else None,
    }

prompt = "In two sentences, explain what a load balancer does."
naive_result = await naive_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt, n=50,
)
print("naive (unbounded):", naive_result)

naive (unbounded): {'n_sent': 50, 'n_ok': 50, 'p95_s': 1.045, 'mean_s': 1.038}


In [31]:
class LoadShedder:
    def __init__(self, max_in_flight: int):
        self.sem = asyncio.Semaphore(max_in_flight)

    async def try_admit(self):
        # non-blocking acquire: True if a slot was free right now, False if not
        acquired = self.sem.locked() is False and self.sem._value > 0
        if acquired:
            await self.sem.acquire()
        return acquired

    def release(self):
        self.sem.release()

async def send_with_shedding(client, shedder, base_url, model, prompt, max_tokens=128):
    admitted = await shedder.try_admit()
    if not admitted:
        return {"ok": False, "shed": True, "latency_s": 0.0}
    try:
        t0 = time.perf_counter()
        r = await client.post(f"{base_url}/v1/chat/completions".replace("/v1/v1", "/v1"),
                                json={"model": model, "messages": [{"role": "user", "content": prompt}],
                                      "max_tokens": max_tokens, "temperature": 0.0})
        r.raise_for_status()
        return {"ok": True, "shed": False, "latency_s": time.perf_counter() - t0}
    except Exception as e:
        return {"ok": False, "shed": False, "latency_s": time.perf_counter() - t0, "error": str(e)}
    finally:
        shedder.release()

async def shedded_burst(base_url, model, prompt, n=50, cap=8, max_tokens=128):
    shedder = LoadShedder(cap)
    async with httpx.AsyncClient(timeout=120.0) as client:
        results = await asyncio.gather(*[
            send_with_shedding(client, shedder, base_url, model, prompt, max_tokens)
            for _ in range(n)
        ])
    accepted = [r for r in results if r["ok"]]
    shed = [r for r in results if r.get("shed")]
    latencies = [r["latency_s"] for r in accepted]
    return {
        "n_sent": n, "cap": cap,
        "n_accepted": len(accepted), "n_shed": len(shed),
        "accepted_p95_s": round(p95(latencies), 3) if latencies else None,
    }

shedded_result = await shedded_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt, n=50, cap=8,
)
print("shedded (cap=8):", shedded_result)

shedded (cap=8): {'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.453}


In [32]:
sweep = []
for n in (8, 16, 32, 50):
    r = await shedded_burst(
        base_url="http://localhost:8000/v1",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        prompt=prompt, n=n, cap=8,
    )
    sweep.append(r)
    print(r)

{'n_sent': 8, 'cap': 8, 'n_accepted': 8, 'n_shed': 0, 'accepted_p95_s': 0.433}
{'n_sent': 16, 'cap': 8, 'n_accepted': 8, 'n_shed': 8, 'accepted_p95_s': 0.408}
{'n_sent': 32, 'cap': 8, 'n_accepted': 8, 'n_shed': 24, 'accepted_p95_s': 0.472}
{'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.458}


In [33]:
import json
report = {
    "naive_unbounded_n50": naive_result,
    "shedded_cap8_n50": shedded_result,
    "shedded_sweep": sweep,
}
with open("shedding_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "naive_unbounded_n50": {
    "n_sent": 50,
    "n_ok": 50,
    "p95_s": 1.045,
    "mean_s": 1.038
  },
  "shedded_cap8_n50": {
    "n_sent": 50,
    "cap": 8,
    "n_accepted": 8,
    "n_shed": 42,
    "accepted_p95_s": 0.453
  },
  "shedded_sweep": [
    {
      "n_sent": 8,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 0,
      "accepted_p95_s": 0.433
    },
    {
      "n_sent": 16,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 8,
      "accepted_p95_s": 0.408
    },
    {
      "n_sent": 32,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 24,
      "accepted_p95_s": 0.472
    },
    {
      "n_sent": 50,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 42,
      "accepted_p95_s": 0.458
    }
  ]
}


In [34]:
#!/usr/bin/env python3
# Green check for the extra W3D3 lab (load shedding under overload).
# Run next to shedding_report.json:  python verify.py
# Prints exactly one line last: GREEN CHECK: PASS  or  GREEN CHECK: FAIL (<reason>)
# stdlib only.
#
# The lab's numbers are live measurements, so there is no ground truth to
# recompute. Instead this holds the report to the invariants any honest run
# satisfies; the margins are deliberately loose so hardware never fails a
# student, only method does.
import json, os
from typing import NoReturn

SHED_FLOOR_N50 = 20        # cap 8, burst 50: most of the burst must be shed
P95_IMPROVEMENT = 0.8      # accepted p95 must be at most 0.8x the naive p95
SWEEP_FLATNESS = 2.5       # accepted p95 across the sweep: max <= 2.5x min


class _Stop(Exception):
    pass


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def need(d, key, where):
    if not isinstance(d, dict) or key not in d:
        _fail("missing %s in %s" % (key, where))
    return d[key]


def main():
    if not os.path.isfile("shedding_report.json"):
        _fail("shedding_report.json not found; run Step 5 first")
    try:
        with open("shedding_report.json") as f:
            r = json.load(f)
    except json.JSONDecodeError as e:
        _fail("shedding_report.json is not valid JSON: %s" % e)

    naive = need(r, "naive_unbounded_n50", "report")
    shed = need(r, "shedded_cap8_n50", "report")
    sweep = need(r, "shedded_sweep", "report")

    if need(naive, "n_sent", "naive") != 50:
        _fail("naive burst must send 50")
    if need(naive, "n_ok", "naive") < 45:
        _fail("only %s/50 naive requests succeeded; fix the baseline before "
              "measuring shedding on top of it" % naive["n_ok"])
    naive_p95 = need(naive, "p95_s", "naive")
    if not isinstance(naive_p95, (int, float)) or naive_p95 <= 0:
        _fail("naive p95 missing or non-positive")

    if need(shed, "cap", "shedded") != 8 or need(shed, "n_sent", "shedded") != 50:
        _fail("the graded shedded burst is n=50 at cap=8")
    n_acc, n_shed = need(shed, "n_accepted", "shedded"), need(shed, "n_shed", "shedded")
    if n_acc + n_shed > 50:
        _fail("accepted (%s) + shed (%s) exceeds the 50 sent" % (n_acc, n_shed))
    if n_acc < 8:
        _fail("only %s accepted at cap 8; the first wave alone should fill the cap" % n_acc)
    if n_shed < SHED_FLOOR_N50:
        _fail("only %s shed out of 50 at cap 8; the shedder is queueing, not "
              "shedding" % n_shed)
    shed_p95 = need(shed, "accepted_p95_s", "shedded")
    if not isinstance(shed_p95, (int, float)) or shed_p95 <= 0:
        _fail("accepted p95 missing or non-positive")
    if shed_p95 >= naive_p95 * P95_IMPROVEMENT:
        _fail("accepted p95 %.2fs vs naive %.2fs: shedding is not protecting "
              "latency by a real margin" % (shed_p95, naive_p95))

    if not isinstance(sweep, list) or [lvl.get("n_sent") for lvl in sweep] != [8, 16, 32, 50]:
        _fail("shedded_sweep must hold the four levels n=8,16,32,50 in order")
    p95s, sheds = [], []
    for lvl in sweep:
        if lvl.get("cap") != 8:
            _fail("sweep level n=%s ran at cap=%s, the sweep fixes cap 8"
                  % (lvl.get("n_sent"), lvl.get("cap")))
        p = lvl.get("accepted_p95_s")
        if not isinstance(p, (int, float)) or p <= 0:
            _fail("sweep level n=%s has no accepted p95" % lvl.get("n_sent"))
        p95s.append(p)
        sheds.append(lvl.get("n_shed", 0))
    if sheds[0] > 1:
        _fail("n=8 at cap 8 shed %s requests; the cap is rejecting inside "
              "capacity" % sheds[0])
    if any(b < a for a, b in zip(sheds, sheds[1:])):
        _fail("shed counts fall as the burst grows (%s); that cannot happen "
              "with a fixed cap" % sheds)
    if sheds[-1] < 10:
        _fail("largest level shed only %s; the cap is not being enforced" % sheds[-1])
    if max(p95s) > min(p95s) * SWEEP_FLATNESS:
        _fail("accepted p95 varies %.2fs..%.2fs across the sweep; the cap is "
              "not holding latency flat" % (min(p95s), max(p95s)))

    print("invariants hold: shedding happened, accepted p95 protected, cap flat")
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    try:
        main()
    except _Stop:
        raise SystemExit(1)


invariants hold: shedding happened, accepted p95 protected, cap flat
GREEN CHECK: PASS


In [35]:
import httpx

try:
    r = httpx.get("http://localhost:8000/v1/models", timeout=5)
    print(r.status_code)
    print(r.json())
except Exception as e:
    print("SERVER NOT RUNNING:", e)

200
{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-1.5B-Instruct', 'object': 'model', 'created': 1788269306, 'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-1.5B-Instruct', 'parent': None, 'max_model_len': 4096, 'permission': [{'id': 'modelperm-d2960852505f47708689ba7753b3f498', 'object': 'model_permission', 'created': 1788269306, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
